# 🧩 Notebook 01b — Generación de embeddings

En esta notebook partimos del DataFrame generado en la notebook anterior (`dialogs-2.0.pkl`) y generamos embeddings para cada turno del corpus.

La notebook:

1. descarga automáticamente el `.pkl` desde Google Drive;
2. carga el DataFrame con los turnos;
3. toma la columna `utterance`;
4. genera embeddings con `sentence-transformers`;
5. guarda los embeddings y los IDs en archivos `.npy`;
6. permite descargar los archivos para usarlos en la siguiente notebook de indexación ANN.

## 1️⃣ Instalación de dependencias

Instalamos `sentence-transformers` para generar embeddings y `gdown` para descargar el archivo `.pkl` directamente desde Google Drive.

In [1]:
!pip install -q sentence-transformers gdown

## 2️⃣ Carga automática del DataFrame

El archivo `dialogs-2.0.pkl` fue generado en la notebook anterior y se encuentra compartido en Google Drive.

En lugar de subirlo manualmente, lo descargamos automáticamente usando el identificador del archivo de Drive. Esto permite que la notebook sea más reproducible y evita depender de la carga manual desde la computadora.

In [2]:
import os
import gdown
import pandas as pd

# Archivo generado por la notebook 01 y compartido desde Google Drive.
# URL original:
# https://drive.google.com/file/d/1kRbmvg3NB96pWMUl866GZRrT6Zq9vxcb/view?usp=sharing

FILE_ID = "1kRbmvg3NB96pWMUl866GZRrT6Zq9vxcb"
ARCHIVO_ENTRADA = "dialogs-2.0.pkl"

# gdown descarga archivos de Google Drive usando una URL directa con el file_id.
# Si el archivo ya existe en el entorno de Colab, no lo vuelve a descargar.
url = f"https://drive.google.com/uc?id={FILE_ID}"

if not os.path.exists(ARCHIVO_ENTRADA):
    gdown.download(url, ARCHIVO_ENTRADA, quiet=False)
else:
    print(f"El archivo {ARCHIVO_ENTRADA} ya existe. No se descarga nuevamente.")

Downloading...
From (original): https://drive.google.com/uc?id=1kRbmvg3NB96pWMUl866GZRrT6Zq9vxcb
From (redirected): https://drive.google.com/uc?id=1kRbmvg3NB96pWMUl866GZRrT6Zq9vxcb&confirm=t&uuid=58f9a446-2b73-48c4-b53a-27b51820beb5
To: /content/dialogs-2.0.pkl
100%|██████████| 156M/156M [00:02<00:00, 62.6MB/s]


In [3]:
df = pd.read_pickle(ARCHIVO_ENTRADA)

print("Shape:", df.shape)
df.head()

Shape: (1000023, 11)


,dataset,split,dialogue_id,turn_id,speaker,utterance,domains,dialog_acts,main_acts,slots,intents
0,ABCD,test,ABCD_test_0,0,user,Hi. My name is Chloe Zhang. I am curious as ...,[storewide query],None,None,None,[timing]
1,ABCD,test,ABCD_test_1,0,user,Hello. I recently signed up for a subscription...,[subscription inquiry],None,None,None,[manage dispute bill]
2,ABCD,test,ABCD_test_1,1,user,"sure, it's Albert Sanders and my account id is...",[subscription inquiry],None,None,[account id],None
3,ABCD,test,ABCD_test_1,2,user,yes its 7149958247,[subscription inquiry],None,None,[order id],None
4,ABCD,test,ABCD_test_2,0,user,I'm thinking about buying an item but first i ...,[single item query],None,None,None,[shirt]


In [4]:
print("Cantidad de datasets incluidos:", df["dataset"].nunique() if "dataset" in df.columns else "No disponible")
print("Cantidad de diálogos incluidos:", df["dialogue_id"].nunique())
print("Cantidad de turnos incluidos:", len(df))

Cantidad de datasets incluidos: 13
Cantidad de diálogos incluidos: 101021
Cantidad de turnos incluidos: 1000023


---

## 3️⃣ Preparación de textos

Tomamos la columna `utterance` como entrada para el modelo de embeddings.

También generamos un arreglo de IDs enteros para asociar cada vector con la fila correspondiente del DataFrame.

In [5]:
import numpy as np

sentences = df["utterance"].fillna("").astype(str).tolist()

ids = np.arange(len(sentences), dtype="int64")
np.save("ids.npy", ids)

print("Cantidad de textos:", len(sentences))
print("Shape de ids:", ids.shape)

Cantidad de textos: 1000023
Shape de ids: (1000023,)


---

## 4️⃣ Generación de embeddings

Usamos modelos preentrenados de `sentence-transformers`.

Para esta primera corrida escalada dejamos activo `all-MiniLM-L6-v2`, porque produce vectores de 384 dimensiones y es más conveniente para trabajar con cientos de miles o millones de turnos en Colab.

Si luego se desea repetir el experimento con `all-mpnet-base-v2`, se puede descomentar la segunda línea del diccionario `model_names`.

In [13]:
from sentence_transformers import SentenceTransformer

model_names = {
    #"minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "mpnet": "sentence-transformers/all-mpnet-base-v2",
}

In [14]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print("✅ GPU detectada — usando CUDA.")
else:
    device = "cpu"
    print("⚠️ No hay GPU — usando CPU.")

✅ GPU detectada — usando CUDA.


In [15]:
BATCH_SIZE = 64

In [16]:
for short_name, model_name in model_names.items():
    print(f"🔹 Generando embeddings con {model_name}")
    model = SentenceTransformer(model_name)

    emb = model.encode(
        sentences,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        device=device
    )

    emb = emb.astype("float32")
    output_file = f"embeddings_{short_name}.npy"
    np.save(output_file, emb)

    print(f"Archivo guardado: {output_file}")
    print(f"Forma de los embeddings ({short_name}):", emb.shape)

    del model
    del emb

    if device == "cuda":
        torch.cuda.empty_cache()

🔹 Generando embeddings con sentence-transformers/all-mpnet-base-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/15626 [00:00<?, ?it/s]

Archivo guardado: embeddings_mpnet.npy
Forma de los embeddings (mpnet): (1000023, 768)


---

## 5️⃣ Verificación rápida

Cargamos los archivos guardados para verificar sus dimensiones.

In [17]:
print("Archivos generados:")
for fname in os.listdir():
    if fname.endswith(".npy"):
        print("-", fname)

Archivos generados:
- ids.npy
- embeddings_minilm.npy
- embeddings_mpnet.npy


In [18]:
ids_check = np.load("ids.npy")
print("ids:", ids_check.shape)

for short_name in model_names.keys():
    fname = f"embeddings_{short_name}.npy"
    emb_check = np.load(fname, mmap_mode="r")
    print(fname, emb_check.shape, emb_check.dtype)

ids: (1000023,)
embeddings_mpnet.npy (1000023, 768) float32


---

## 6️⃣ Descarga de archivos

Descargamos los embeddings y los IDs para continuar con la notebook de indexación y evaluación ANN.

In [19]:
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

carpeta = "/content/drive/MyDrive/Doctorado/Cursos/ANN/TF/data/version2.0"
os.makedirs(carpeta, exist_ok=True)

shutil.copy("ids.npy", carpeta)

for short_name in model_names.keys():
    shutil.copy(f"embeddings_{short_name}.npy", carpeta)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
